|name|tags|url|
|---|---|---|
|Introduction to Apache Airflow in Python|python, airflow|https://app.datacamp.com/learn/courses/introduction-to-apache-airflow-in-python|

# 1/ Intro to Airflow
---

> Workflow: Tập hợp các bước để hoàn thành 1 nhiệm vụ liên quan đến dữ liệu
> 
> Airflow: 1 công cụ dùng để tổ chức workflow
>
> Ngoài ra còn có 1 số công cụ khác: Luigi, SSIS, Bash scripting

# DAGs
---
> Workflow được cài đặt dưới dạng DAGs (Directed Acyclic Graphs). Bao gồm các task và sự phụ thuộc giữa các task
>> Có hướng: Thể hiện sự lệ thuộc giữa các tasks
>
>> Không chu trình: Các task không thể được lặp lại = chỉ thực hiện 1 lần duy nhất

## 2/ Implementing Airflow DAGs
---

## Dags CLI
```console
$ airflow dags list-import-error: Hiển thị lỗi
```

In [ ]:
# Example
etl_dag = DAG(
    dag_id="etl_pipeline",
    default_args={"start_date": "2024-01-08"}
)

## Running a workflow in Airflow
```console
$ airflow tasks test  <dag_id> <task_id> [execution_date]
```

## Define a DAG
---

In [ ]:
from airflow import DAG
from datetime import datetime

# Attribute apply to DAG
default_arguments = {
    "owner": "jdoe",
    "email": "jdoe@datacamp.com",
    "start_date": datetime(2020,1,20)
}

# Method 1
etl_dag = DAG("etl_workflow",  default_arguments=default_arguments )

# Method 2
with DAG("etl_workflow", default_args=default_arguments) as etl_dag:
    

### Working with DAGs with shell
---
```console
$ airflow -h // Xem option
$ airflow task -h // Xem option
```

### Start airflow webserver on port
---
```console
$ airflow webservice -p 9090
```

### Dag Runs
---
Là thể hiện của workflow tại 1 lần chạy 

Preset:
@hourly -> 0 * * * * 
@daily -> 0 0 * * * 
@weekly -> 0 0 * * 0
@none
@once

<b>real_start_time = start_date + schedule_interval</b>

## Operator
---
Đại diện cho task trong DAG

### Bash Operator
---

In [ ]:
# Import libraries
from airflow.operators.bash import BashOperator

# Execute bash command or bash script
BashOperator(
    task_id="bash_example",
    bash_command='echo "Exapmple!"',
    # Next line only for Airflow < 2.0
    dag=dag
)

BashOperator(
    task_id='bash_script_example'
    bash_commmand='runcleanup.sh'
)

In [ ]:
# More example
from airflow.operators.bash import BashOperator

example_task = BashOperator(task_id='bash_ex',
                            bash_commadn='echo 1',
                           )

bash_task = BashOperator(task_id='clean_adresses',
                         bash_command='cat addresses.txt | awk "NF==10" > cleaned.txt'
                        )

### PythonOperator: To run python function
---

In [ ]:
# Import libraries
from airflow.operators.python import PythonOperator

def printme():
    print("This goes in logs!")

python_task = PythonOperator(
    task_id="simple_print",
    python_callable=printme
)

In [ ]:
# Add argument to callable
def sleep(length_of_time):
    time.sleep(length_of_time)

sleep_task = PythonOperator(
    task_id="sleep",
    python_callable=sleep,
    op_kwargs={"length_of_time": 5} # Keyword args
)

## EmailOperator: To send email from airflow
---

In [ ]:
# Import libraries
from airflow.operators.email import EmailOperator

email_task = EmailOperator(
    task_id="email_sales_report",
    to="sales_manager@example.com",
    subject="Autoomated Sales Report",
    html_content="Attached is the latest sales report",
    files="latest_sales.xlsx"
)

## Airflow Tasks
---
Là thể hiện của Operator. Dùng để tham chiếu đến Operator ( không dùng biến)

### Simple task dependency
---

In [ ]:
# Define the tasks
task1 = BashOperator(task_id="first_task",
                     bash_command="echo 1")

task2 = BashOperator(task_id="second_task",
                     bash_command="echo 2")

task1 >> task2 # Do task 1 before task 2

In [ ]:
# Exercise

## 3/ Maintaining and monitoring Airflow workflows
---

### Sensor:
- Là 1 toán tử đặc biệt, dùng để kiểm tra điều kiện được thỏa
- Dùng khi không biết khi nào đúng và tránh tạo chu kỳ

### FileSensor: Kiểm tra sự tồn tại của file
---

In [ ]:
# Import libraries
from airflow.sensors.filesystem import FileSensor

file_sensor_task = FileSensor(task_id="file_sense",
                              filepath="salesdate.csv",
                              poke_interval=300, # Khoảng thời gian giữa các lần check
                              dag=sales_report_dag)

# Dag dependency
init_sales_cleanup >> file_sensor_task >> generate_report

### ExternalTaskSensor: Chờ DAG khác hoàn thành
### HttpSensor: Gửi request đến url và check content của response
### SqlSensor: Truy vấn SQL và check content trả về
---

### Determining your executor
---
```console
// C1:
$ cat airflow/airflow.cfg | grep "executor = "
// C2:
$ airflow info
```

### SLAs
---
Là khoảng thời gian là mình mong muốn cái DAG hoàn thành. Nếu không hoàn thành sẽ gửi email về.

In [ ]:
from airflow.operator.bash import BashOperator
from datetime import datatime, timedelta

# Task sla
task1 = BashOperator(task_id="sla_task",
                     bash_command="runcode.sh"
                     sla=timedelta(seconds=30),
                     dag=dag)

# Dag sla:
default_args = {
    "sla": timedelta(minutes=20), # Sử dụng đối tượng timedelta của datetime module
    "start_date": datetime(2026,2,28),
    "email": ["airflowalert@datacamp.com"],
    "email_on_failure": True, # Gửi email khi thất bại
    "email_on_retry": False,
    "email_on_success": True,
}

with DAG("sla_dag", default_args=default_args) as dag:
    pass